In [5]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import Aer
from itertools import product


# =========================
# HELPER
# =========================
def initialize_bits(qc, reg, bitstring):
    for i, b in enumerate(reversed(bitstring)):
        if b == '1':
            qc.x(reg[i])


# =========================
# ROUTER WITH FULL BARRIERS
# =========================
def router(D_val, N1_val, N2_val):

    D  = QuantumRegister(3, 'D')
    N1 = QuantumRegister(3, 'N1')
    N2 = QuantumRegister(3, 'N2')

    X1 = QuantumRegister(3, 'X1')
    X2 = QuantumRegister(3, 'X2')

    d1 = QuantumRegister(2, 'd1')
    d2 = QuantumRegister(2, 'd2')

    anc = QuantumRegister(9, 'anc')
    flag = QuantumRegister(1, 'flag')

    OUT = QuantumRegister(3, 'OUT')
    c = ClassicalRegister(3, 'c')

    qc = QuantumCircuit(D, N1, N2, X1, X2, d1, d2, anc, flag, OUT, c)

    # INPUT
    qc.barrier(label="INPUT")
    initialize_bits(qc, D, D_val)
    initialize_bits(qc, N1, N1_val)
    initialize_bits(qc, N2, N2_val)

    # XOR
    qc.barrier(label="XOR")
    for i in range(3):
        qc.cx(N1[i], X1[i])
        qc.cx(D[i],  X1[i])

        qc.cx(N2[i], X2[i])
        qc.cx(D[i],  X2[i])

    # POPCOUNT
    qc.barrier(label="POPCOUNT")

    # d1
    qc.cx(X1[0], d1[0])
    qc.cx(X1[1], d1[0])
    qc.cx(X1[2], d1[0])

    qc.ccx(X1[0], X1[1], anc[0])
    qc.ccx(X1[0], X1[2], anc[1])
    qc.ccx(X1[1], X1[2], anc[2])

    qc.cx(anc[0], d1[1])
    qc.cx(anc[1], d1[1])
    qc.cx(anc[2], d1[1])

    # d2
    qc.cx(X2[0], d2[0])
    qc.cx(X2[1], d2[0])
    qc.cx(X2[2], d2[0])

    qc.ccx(X2[0], X2[1], anc[3])
    qc.ccx(X2[0], X2[2], anc[4])
    qc.ccx(X2[1], X2[2], anc[5])

    qc.cx(anc[3], d2[1])
    qc.cx(anc[4], d2[1])
    qc.cx(anc[5], d2[1])

    # COMPARATOR
    qc.barrier(label="COMPARATOR (d1 > d2)")

    qc.x(d2[1])
    qc.ccx(d1[1], d2[1], flag[0])
    qc.x(d2[1])

    qc.cx(d1[1], anc[6])
    qc.cx(d2[1], anc[6])
    qc.x(anc[6])

    qc.x(d2[0])
    qc.ccx(d1[0], d2[0], anc[7])
    qc.x(d2[0])

    qc.ccx(anc[6], anc[7], anc[8])
    qc.cx(anc[8], flag[0])

    # SELECTION
    qc.barrier(label="SELECTION")

    for i in range(3):
        qc.ccx(flag[0], N2[i], OUT[i])
        qc.x(flag[0])
        qc.ccx(flag[0], N1[i], OUT[i])
        qc.x(flag[0])

    # UNCOMPUTE COMPARATOR
    qc.barrier(label="UNCOMPUTE COMPARATOR")

    qc.cx(anc[8], flag[0])
    qc.ccx(anc[6], anc[7], anc[8])

    qc.x(d2[0])
    qc.ccx(d1[0], d2[0], anc[7])
    qc.x(d2[0])

    qc.x(anc[6])
    qc.cx(d2[1], anc[6])
    qc.cx(d1[1], anc[6])

    qc.x(d2[1])
    qc.ccx(d1[1], d2[1], flag[0])
    qc.x(d2[1])

    # UNCOMPUTE POPCOUNT
    qc.barrier(label="UNCOMPUTE POPCOUNT")

    qc.cx(anc[5], d2[1])
    qc.cx(anc[4], d2[1])
    qc.cx(anc[3], d2[1])

    qc.ccx(X2[1], X2[2], anc[5])
    qc.ccx(X2[0], X2[2], anc[4])
    qc.ccx(X2[0], X2[1], anc[3])

    qc.cx(X2[2], d2[0])
    qc.cx(X2[1], d2[0])
    qc.cx(X2[0], d2[0])

    qc.cx(anc[2], d1[1])
    qc.cx(anc[1], d1[1])
    qc.cx(anc[0], d1[1])

    qc.ccx(X1[1], X1[2], anc[2])
    qc.ccx(X1[0], X1[2], anc[1])
    qc.ccx(X1[0], X1[1], anc[0])

    qc.cx(X1[2], d1[0])
    qc.cx(X1[1], d1[0])
    qc.cx(X1[0], d1[0])

    # UNDO XOR
    qc.barrier(label="UNDO XOR")

    for i in range(3):
        qc.cx(D[i],  X2[i])
        qc.cx(N2[i], X2[i])

        qc.cx(D[i],  X1[i])
        qc.cx(N1[i], X1[i])

    # MEASURE
    qc.barrier(label="MEASURE")

    for i in range(3):
        qc.measure(OUT[i], c[i])

    return qc


# =========================
# DRAW CIRCUIT (MPL)
# =========================
qc = router("000", "001", "111")

fig = qc.draw('mpl', fold=120, idle_wires=False)


In [4]:
  qc.draw('mpl', fold=120)

NameError: name 'qc' is not defined